In [0]:
%run ./schemas

In [0]:
%run ./common

In [0]:
# Cell 3: Widgets — fill these out BEFORE running the remainder.
# Defaults are set to your already-downloaded incremental batch.
dbutils.widgets.text("batch_id", "incremental_20261008_001")
dbutils.widgets.text("run_date", "2026-10-08")
dbutils.widgets.dropdown("load_type", "INCREMENTAL", ["INCREMENTAL", "FULL"])
dbutils.widgets.text("manifest_path", "")

print("Parameters ready. Check the widgets at the top of the notebook.")

In [0]:
# Cell 4: Imports, configuration and manifest validation.
import json
import re
import time
import uuid
import traceback
from datetime import date
from pathlib import Path

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

batch_id = dbutils.widgets.get("batch_id").strip()
run_date = dbutils.widgets.get("run_date").strip()
load_type = dbutils.widgets.get("load_type").strip().upper()
manifest_override = dbutils.widgets.get("manifest_path").strip()

# Keep explicit paths; do not rely on current_date() or implicit file discovery.
if not batch_id or not re.fullmatch(r"[A-Za-z0-9_-]{1,100}", batch_id):
    raise ValueError("Set a valid batch_id using the notebook widget")
if not run_date or date.fromisoformat(run_date).isoformat() != run_date:
    raise ValueError("Set run_date as YYYY-MM-DD")
if load_type not in ("FULL", "INCREMENTAL"):
    raise ValueError("load_type must be FULL or INCREMENTAL")

manifest_dir = Path(ROOT) / "_manifests"
if manifest_override:
    manifest_file = Path(manifest_override)
elif load_type == "INCREMENTAL":
    manifest_file = manifest_dir / f"incremental_{batch_id}.json"
else:
    matches = list(manifest_dir.glob(f"full_load_*_{batch_id}.json"))
    if len(matches) != 1:
        raise ValueError("FULL manifest not found uniquely; set manifest_path widget")
    manifest_file = matches[0]

if not manifest_file.is_file():
    raise FileNotFoundError(f"Cannot find manifest: {manifest_file}")
manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
if manifest.get("batch_id") != batch_id:
    raise ValueError("Manifest batch_id does not match notebook widget")
if manifest.get("load_type") != load_type:
    raise ValueError("Manifest load_type does not match notebook widget")
if manifest.get("run_date") and manifest["run_date"] != run_date:
    raise ValueError("Manifest run_date does not match notebook widget")

expected = {(r, e) for r in REPOS for e in ENTITIES}
files = [x for x in manifest["files"] if (x.get("repository"), x.get("entity")) in expected]
seen = [(x["repository"], x["entity"]) for x in files]
if len(files) != len(expected) or set(seen) != expected or len(set(seen)) != len(seen):
    raise ValueError("Manifest must list exactly 9 distinct repo/entity JSONL files")

# Deterministic order: all Bronze, then all Silver.
repo_order = {name: index for index, name in enumerate(REPOS)}
entity_order = {name: index for index, name in enumerate(ENTITIES)}
files.sort(key=lambda x: (repo_order[x["repository"]], entity_order[x["entity"]]))
root_prefix = str(Path(ROOT)) + "/"
for item in files:
    p = Path(item["path"])
    if not str(p).startswith(root_prefix) or p.suffix != ".jsonl":
        raise ValueError(f"Unexpected source path in manifest: {p}")
    if not p.is_file():
        raise FileNotFoundError(p)

print("Ready to process:", batch_id, load_type)
print("Manifest:", manifest_file)
print("Files:", len(files), "| Raw manifest records:", sum(int(x.get("records", 0)) for x in files))
for x in files:
    print("  ", x["repository"], x["entity"], x.get("records", "?"), "rows")

In [0]:
# Cell 5: Single-session transformations. Explicit schemas and same MERGE semantics
# as your working notebooks. No notebook.run, no cache, no serverless-blocked config.

BUSINESS_COLS = {
    "commits": ["sha", "node_id", "commit", "author", "committer", "html_url"],
    "issues": ["id", "number", "node_id", "title", "state", "created_at",
               "updated_at", "closed_at", "user", "labels", "html_url"],
    "pulls": ["id", "number", "node_id", "title", "state", "draft",
              "created_at", "updated_at", "closed_at", "merged_at",
              "user", "merged", "html_url"],
}


def merge_delta_changed(df, table_name):
    """Idempotent upsert that does not replay stale historical versions.

    A changed payload updates a matched key ONLY if its GitHub source timestamp
    is strictly newer than the stored one. Missing/equal timestamps do not
    overwrite a stored record. Unchanged rows retain load_timestamp/batch_id.
    Schema evolution remains an explicit, separate concern.
    """
    columns = df.columns
    values = {column: f"s.`{column}`" for column in columns}
    updates = {column: expression for column, expression in values.items()
               if column != "record_key"}
    (DeltaTable.forName(spark, table_name).alias("t")
        .merge(df.alias("s"), "t.record_key = s.record_key")
        .whenMatchedUpdate(
            condition=(
                "NOT (t.payload_hash <=> s.payload_hash) "
                "AND s.source_updated_at IS NOT NULL "
                "AND (t.source_updated_at IS NULL OR s.source_updated_at > t.source_updated_at)"
            ),
            set=updates
        )
        .whenNotMatchedInsert(values=values)
        .execute())
    metrics = spark.sql(f"DESCRIBE HISTORY {table_name} LIMIT 1").first()["operationMetrics"]
    return (int(metrics.get("numTargetRowsInserted", 0)),
            int(metrics.get("numTargetRowsUpdated", 0)))


def source_keys_from_raw(raw, repo, entity):
    """Keys requested in *this* raw batch, including unchanged previously-loaded keys."""
    id_col = F.col("sha") if entity == "commits" else F.col("id").cast("string")
    good = (
        raw.filter(F.col("_corrupt_record").isNull())
           .filter(id_col.isNotNull() & (F.trim(id_col) != ""))
           .filter(F.col("_repo").isNull() | (F.col("_repo") == repo))
    )
    if entity == "issues":
        good = good.filter(F.col("pull_request").isNull())
    return good.select(
        F.concat_ws("|", F.lit(repo), id_col).alias("record_key")
    ).distinct()


def run_bronze_one(item):
    repo, entity, file_path = item["repository"], item["entity"], item["path"]
    started, execution_id = now(), str(uuid.uuid4())
    count = 0
    t0 = time.perf_counter()
    try:
        raw = (spark.read.schema(RAW_SCHEMAS[entity])
               .option("mode", "PERMISSIVE")
               .option("columnNameOfCorruptRecord", "_corrupt_record")
               .json(file_path))
        count = raw.count()
        raw_id = F.col("sha") if entity == "commits" else F.col("id").cast("string")
        updated = F.col("commit.author.date") if entity == "commits" else F.col("updated_at")
        reason = (F.when(F.col("_corrupt_record").isNotNull(), "BAD_JSON_OR_TYPE")
                  .when(raw_id.isNull() | (F.trim(raw_id) == ""), "MISSING_PRIMARY_ID")
                  .when(F.col("_repo").isNotNull() & (F.col("_repo") != repo),
                        "REPOSITORY_MISMATCH"))
        if entity == "issues":
            reason = reason.when(F.col("pull_request").isNotNull(), "PULL_REQUEST_IN_ISSUES")
        staged = (raw.withColumn("_reason", reason)
                  .withColumn("_entity_id", raw_id)
                  .withColumn("_updated_str", updated))

        payload_cols = [F.col(c) for c in RAW_SCHEMAS[entity].fieldNames()
                        if c != "_corrupt_record"]
        failed = staged.filter(F.col("_reason").isNotNull())
        rejected_payload = F.coalesce(
            F.col("_corrupt_record"), F.to_json(F.struct(*payload_cols))
        )
        quarantined_df = failed.select(
            F.sha2(F.concat_ws("|", F.lit(batch_id), F.lit(repo), F.lit(entity),
                               rejected_payload), 256).alias("quarantine_id"),
            F.lit(batch_id).alias("batch_id"),
            F.lit(repo).alias("repository"),
            F.lit(entity).alias("entity"),
            F.lit(file_path).alias("source_file"),
            rejected_payload.alias("raw_payload"),
            F.col("_reason").alias("error_type"),
            F.concat(F.lit("Failed Bronze validation: "),
                     F.col("_reason")).alias("error_reason"),
            F.current_timestamp().alias("load_timestamp"),
            F.current_timestamp().alias("quarantined_at"),
        )
        quarantined_count = quarantine(spark, quarantined_df, "bronze")

        valid = staged.filter(F.col("_reason").isNull())
        relevant_json = F.to_json(F.struct(*[F.col(c) for c in BUSINESS_COLS[entity]]))
        full_payload = F.to_json(F.struct(*payload_cols))
        source = valid.select(
            F.concat_ws("|", F.lit(repo), F.col("_entity_id")).alias("record_key"),
            F.lit(repo).alias("repository"),
            F.lit(batch_id).alias("batch_id"),
            F.lit(file_path).alias("source_file"),
            F.expr("try_cast(_updated_str as timestamp)").alias("source_updated_at"),
            full_payload.alias("raw_payload"),
            F.sha2(relevant_json, 256).alias("payload_hash"),
            F.current_timestamp().alias("load_timestamp"),
            F.expr("try_cast(_ingested_at as timestamp)").alias("ingested_at"),
        )
        ranking = Window.partitionBy("record_key").orderBy(
            F.col("source_updated_at").desc_nulls_last(),
            F.col("payload_hash").desc()
        )
        source = (source.withColumn("_rn", F.row_number().over(ranking))
                  .filter(F.col("_rn") == 1).drop("_rn"))
        valid_count = source.count()
        inserted, updated_count = (merge_delta_changed(source, f"{CATALOG}.bronze.{entity}")
                                   if valid_count else (0, 0))
        result = {"repository": repo, "entity": entity, "rows_processed": count,
                  "valid_keys": valid_count, "inserted": inserted,
                  "updated": updated_count, "upserted": inserted + updated_count,
                  "quarantined": quarantined_count}
        log(spark, execution_id, batch_id, "raw_to_bronze", "BRONZE",
            run_date, load_type, started, "SUCCESS", count,
            upserted=result["upserted"], quarantined=quarantined_count)
        print(f"BRONZE {repo:30} {entity:8} input={count:<6} "
              f"inserted={inserted:<5} updated={updated_count:<5} "
              f"quarantine={quarantined_count} ({time.perf_counter()-t0:.1f}s)", flush=True)
        # Returning raw input key set permits Silver retry even when Bronze MERGE is a no-op.
        return result, source_keys_from_raw(raw, repo, entity)
    except Exception:
        try:
            log(spark, execution_id, batch_id, "raw_to_bronze", "BRONZE",
                run_date, load_type, started, "FAILED", count, traceback.format_exc())
        except Exception as exc:
            print("BRONZE audit failure:", repr(exc), flush=True)
        raise


def run_silver_one(item, current_batch_keys):
    repo, entity = item["repository"], item["entity"]
    started, execution_id = now(), str(uuid.uuid4())
    count = 0
    t0 = time.perf_counter()
    try:
        # Read latest corresponding Bronze records by the RAW batch's business keys.
        # A no-op Bronze retry retains its original batch_id, so filtering ONLY
        # Bronze.batch_id would incorrectly skip it.
        bronze = (spark.table(f"{CATALOG}.bronze.{entity}")
                  .filter(F.col("repository") == repo)
                  .join(current_batch_keys, "record_key", "left_semi"))
        count = bronze.count()
        if count == 0:
            result = {"repository": repo, "entity": entity, "rows_processed": 0,
                      "valid_keys": 0, "inserted": 0, "updated": 0,
                      "upserted": 0, "quarantined": 0}
            log(spark, execution_id, batch_id, "bronze_to_silver", "SILVER",
                run_date, load_type, started, "SUCCESS", 0)
            print(f"SILVER {repo:30} {entity:8} no valid keys ({time.perf_counter()-t0:.1f}s)", flush=True)
            return result

        parsed = bronze.withColumn("api", F.from_json("raw_payload", RAW_SCHEMAS[entity]))
        ts = lambda expr: F.expr(f"try_cast({expr} as timestamp)")
        if entity == "commits":
            silver = parsed.select(
                "record_key", "repository", F.col("api.sha").alias("commit_sha"),
                F.col("api.author.id").alias("author_user_id"),
                ts("api.commit.author.date").alias("commit_timestamp"),
                F.col("api.commit.message").alias("message"),
                "batch_id", "source_updated_at", "payload_hash",
                F.current_timestamp().alias("load_timestamp"))
            reason = (F.when(F.col("commit_sha").isNull(), "NULL_COMMIT_SHA")
                      .when(F.col("commit_timestamp").isNull(), "INVALID_COMMIT_TIMESTAMP"))
        elif entity == "issues":
            silver = parsed.select(
                "record_key", "repository", F.col("api.id").alias("issue_id"),
                F.col("api.number").alias("issue_number"),
                F.col("api.user.id").alias("author_user_id"),
                F.col("api.title").alias("title"), F.col("api.state").alias("state"),
                ts("api.created_at").alias("created_at"),
                ts("api.updated_at").alias("updated_at"),
                ts("api.closed_at").alias("closed_at"),
                "batch_id", "source_updated_at", "payload_hash",
                F.current_timestamp().alias("load_timestamp"))
            reason = (F.when(F.col("issue_id").isNull(), "NULL_ISSUE_ID")
                      .when(F.col("created_at").isNull() | F.col("updated_at").isNull(),
                            "INVALID_TIMESTAMP"))
        else:
            silver = parsed.select(
                "record_key", "repository", F.col("api.id").alias("pr_id"),
                F.col("api.number").alias("pr_number"),
                F.col("api.user.id").alias("author_user_id"),
                F.col("api.title").alias("title"), F.col("api.state").alias("state"),
                F.col("api.draft").alias("draft"),
                ts("api.created_at").alias("created_at"),
                ts("api.updated_at").alias("updated_at"),
                ts("api.closed_at").alias("closed_at"),
                ts("api.merged_at").alias("merged_at"),
                "batch_id", "source_updated_at", "payload_hash",
                F.current_timestamp().alias("load_timestamp"))
            reason = (F.when(F.col("pr_id").isNull(), "NULL_PR_ID")
                      .when(F.col("created_at").isNull() | F.col("updated_at").isNull(),
                            "INVALID_TIMESTAMP"))

        staged = silver.withColumn("_reason", reason)
        failed = staged.filter(F.col("_reason").isNotNull())
        q = (failed.join(bronze.select("record_key", "raw_payload"), "record_key", "left")
             .select(
                F.sha2(F.concat_ws("|", F.lit(batch_id), F.lit(entity),
                                   "record_key", "_reason"), 256).alias("quarantine_id"),
                F.lit(batch_id).alias("batch_id"), F.col("repository"),
                F.lit(entity).alias("entity"), F.col("record_key"), F.col("raw_payload"),
                F.col("_reason").alias("error_type"),
                F.concat(F.lit("Silver validation failure: "),
                         F.col("_reason")).alias("error_reason"),
                F.current_timestamp().alias("load_timestamp"),
                F.current_timestamp().alias("quarantined_at"),
             ))
        quarantined_count = quarantine(spark, q, "silver")
        valid = staged.filter(F.col("_reason").isNull()).drop("_reason")
        ranking = Window.partitionBy("record_key").orderBy(
            F.col("source_updated_at").desc_nulls_last(), F.col("payload_hash").desc())
        valid = (valid.withColumn("_rn", F.row_number().over(ranking))
                 .filter(F.col("_rn") == 1).drop("_rn"))
        valid_count = valid.count()
        inserted, updated_count = (merge_delta_changed(valid, f"{CATALOG}.silver.{entity}")
                                   if valid_count else (0, 0))
        result = {"repository": repo, "entity": entity, "rows_processed": count,
                  "valid_keys": valid_count, "inserted": inserted,
                  "updated": updated_count, "upserted": inserted + updated_count,
                  "quarantined": quarantined_count}
        log(spark, execution_id, batch_id, "bronze_to_silver", "SILVER",
            run_date, load_type, started, "SUCCESS", count,
            upserted=result["upserted"], quarantined=quarantined_count)
        print(f"SILVER {repo:30} {entity:8} input={count:<6} "
              f"inserted={inserted:<5} updated={updated_count:<5} "
              f"quarantine={quarantined_count} ({time.perf_counter()-t0:.1f}s)", flush=True)
        return result
    except Exception:
        try:
            log(spark, execution_id, batch_id, "bronze_to_silver", "SILVER",
                run_date, load_type, started, "FAILED", count, traceback.format_exc())
        except Exception as exc:
            print("SILVER audit failure:", repr(exc), flush=True)
        raise

In [0]:
# Cell 6: EXECUTE. All Bronze first, then all Silver. One Spark session, no child runs.
# Click Run All once with the widgets set to your desired batch.

overall_start = now()
overall_execution_id = str(uuid.uuid4())
bronze_results, silver_results = [], []
keys_by_pair = {}
processed = 0
try:
    ensure_entity_tables(spark)
    print("\n=========== BRONZE — 9 FILES ===========", flush=True)
    for item in files:
        result, keys = run_bronze_one(item)
        bronze_results.append(result)
        keys_by_pair[(item["repository"], item["entity"])] = keys
        processed += result["rows_processed"]

    print("\n=========== SILVER — 9 FILES ===========", flush=True)
    for item in files:
        pair = (item["repository"], item["entity"])
        result = run_silver_one(item, keys_by_pair[pair])
        silver_results.append(result)
        processed += result["rows_processed"]

    log(spark, overall_execution_id, batch_id, "driver_single_session",
        "PIPELINE", run_date, load_type, overall_start, "SUCCESS", processed,
        upserted=sum(x["upserted"] for x in bronze_results + silver_results),
        quarantined=sum(x["quarantined"] for x in bronze_results + silver_results))
except Exception:
    error = traceback.format_exc()
    print("\nPIPELINE FAILED:\n" + error, flush=True)
    try:
        log(spark, overall_execution_id, batch_id, "driver_single_session",
            "PIPELINE", run_date, load_type, overall_start, "FAILED", processed,
            error=error,
            upserted=sum(x["upserted"] for x in bronze_results + silver_results),
            quarantined=sum(x["quarantined"] for x in bronze_results + silver_results))
    except Exception as audit_error:
        print("PIPELINE AUDIT FAILED:", repr(audit_error), flush=True)
    raise

summary = {
    "status": "SUCCESS",
    "batch_id": batch_id,
    "load_type": load_type,
    "bronze_processed": sum(x["rows_processed"] for x in bronze_results),
    "bronze_upserted": sum(x["upserted"] for x in bronze_results),
    "silver_processed": sum(x["rows_processed"] for x in silver_results),
    "silver_upserted": sum(x["upserted"] for x in silver_results),
    "bronze_quarantined": sum(x["quarantined"] for x in bronze_results),
    "silver_quarantined": sum(x["quarantined"] for x in silver_results),
}
print("\n=========== COMPLETE ===========", flush=True)
print(json.dumps(summary, indent=2), flush=True)
# Do not call dbutils.notebook.exit here: leaving output visible for manual Run All.